# The algorithm

This notebook explains how `match` allocates the cleaned choices from [Preprocessing](02-preprocessing.ipynb) to projects, and demonstrates it end-to-end on the real dataset.

We start by reproducing the cleaned choices:

In [ ]:
import pathlib

import pandas as pd

import matched

raw = pathlib.Path("./raw")

projects = pd.read_csv(raw / "projects.csv").set_index("code")
choices_wide = pd.read_csv(raw / "internal-choices.csv").set_index("username")
choice_cols = [c for c in choices_wide.columns if c.startswith("choice")]

choices = {
    username: [code for code in row[choice_cols] if pd.notna(code)]
    for username, row in choices_wide.iterrows()
}
scores = choices_wide.score.to_dict()
courses = choices_wide.course.to_dict()
nmax = projects.nmax.to_dict()
eligible_courses = {
    code: [c for c in ("course1", "course2") if row[c]]
    for code, row in projects.iterrows()
}

clean_choices = matched.filter_invalid_code(choices, nmax)
clean_choices = matched.filter_invalid_course(clean_choices, courses, eligible_courses)
clean_choices = matched.deduplicate(clean_choices)

len(clean_choices)

### How allocation works

`match` allocates in rounds. In each round, it looks at every unallocated student's current *best remaining* choice (round 1: everyone's first choice; round 2: the first choice of whoever wasn't allocated yet, second choice for the rest; and so on). For each project appearing in that round:

- Students who picked it are ranked by `score`, highest first.
- They are admitted until the project's `nmax` capacity is reached.
- Once a project is full, it is removed from consideration for the rest of the round - so students who miss out fall through to their next choice in a later round.

A student's score only matters as a tiebreaker *when a project they've picked is oversubscribed at that round* - it never overrides preference order.

### A closer look: `phii-010`

`phii-010` has `nmax=2` and is picked by several students at different ranks - a good project to see the mechanics on:

In [ ]:
matched.shortlist(clean_choices, scores, "phii-010")

Only two students picked it as their first choice (`mn234` and `gh878`), so both are admitted in round 1 without any tiebreak - capacity isn't actually tested here. To see what happens when a project *is* oversubscribed, suppose `phii-010` could only take a single student instead of two:

In [ ]:
nmax_scarce = dict(nmax)
nmax_scarce["phii-010"] = 1

allocated_scarce = matched.match(clean_choices, scores, nmax_scarce)
ranks_scarce = matched.choice_rank(clean_choices, allocated_scarce)

{u: (allocated_scarce[u], ranks_scarce[u]) for u in ("mn234", "gh878")}

`mn234` (score 68.0) is admitted over `gh878` (47.9) even though both ranked `phii-010` first - the higher score wins the tiebreak. `gh878` isn't dropped entirely: they fall through to `osca-166`, their next remaining valid choice (rank 2).

### The real allocation

With the real capacities, here is the full allocation:

In [ ]:
allocated = matched.match(clean_choices, scores, nmax)
ranks = matched.choice_rank(clean_choices, allocated)

pd.DataFrame({"code": allocated, "choice": ranks}).sort_index()

Most students get their first choice, but not everyone: `kl112` (score 59.6) loses `xiio-044` to `cd634` (62.1) and is allocated `indi-100`, their second choice, instead - the same round-robin/tiebreak mechanism from the `phii-010` example, now playing out on the real cohort. Everyone is still allocated here; a student who couldn't be placed at all would show `None` for both `code` and `choice` (via `allocated[username]` and `ranks[username]`).